# T1.3 (complemento) - Profiling automático com ydata-profiling

Sugestão do professor: gerar um `ProfileReport` do `diabetic_data.csv`. Complementa o `01_eda.ipynb` (EDA orientada a hipóteses) com um varrimento automático das 50 colunas: estatísticas por variável, missing, correlações e alertas (colunas constantes, desequilíbrio, alta cardinalidade).

**Ambiente:** o `ydata-profiling` ainda exige pandas < 3, por isso este notebook corre no `.venv-profiling` (kernel *Bootcamp (profiling)*), não no `.venv` principal. Ver README. O pacote avisa que `ydata_profiling` foi descontinuado a favor de `fg-data-profiling`; mantém-se o nome sugerido pelo professor e o aviso é silenciado.

Os dados são lidos com `src/data/load.py` (só `?` é missing; `"None"` em `A1Cresult`/`max_glu_serum` = teste não realizado). As secções de amostras e de linhas duplicadas estão desligadas: só se mostram agregados, nunca linhas individuais. O HTML gerado fica em `reports/` e não é versionado.

In [1]:
import contextlib
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import warnings

import pandas as pd

with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)
    from ydata_profiling import ProfileReport

from src.data.load import load_diabetic_data, validate_diabetic_data
from src.data.mappings import EXPECTED_MAPPINGS

REPORT_DIR = ROOT / "reports"
REPORT_DIR.mkdir(exist_ok=True)

df = load_diabetic_data(ROOT / "data" / "raw" / "diabetic_data.csv")
validate_diabetic_data(df)
df.shape

(101766, 50)

## Preparação mínima para o perfil

- Os 3 IDs (`admission_type_id`, `discharge_disposition_id`, `admission_source_id`) são códigos, não quantidades: passam a categóricas para não terem média nem correlação de Pearson.
- `target` = `readmitted == "<30"` (P9 em `docs/decisions.md`).
- `encounter_id` e `patient_nbr` são identificadores: ficam fora do perfil (as contagens já são validadas por `validate_diabetic_data`).
- População completa (101 766 encontros, incluindo óbito/hospício), como no `01_eda` (P11).

In [2]:
prof = df.drop(columns=["encounter_id", "patient_nbr"])
for col in EXPECTED_MAPPINGS:
    prof[col] = prof[col].astype("category")
prof["target"] = (prof["readmitted"] == "<30").astype(int)
prof.shape

(101766, 49)

## Relatório

Configuração para ~100 mil linhas: só a correlação `auto` (escolhe a métrica conforme o tipo das variáveis), sem interações (pares de scatter) e sem heatmap de missing. Demora alguns minutos.

In [3]:
profile = ProfileReport(
    prof,
    title="Diabetes 130-US - profiling dos dados brutos",
    samples=None,
    duplicates=None,
    interactions=None,
    correlations={
        "auto": {"calculate": True},
        "pearson": {"calculate": False},
        "spearman": {"calculate": False},
        "kendall": {"calculate": False},
        "phi_k": {"calculate": False},
        "cramers": {"calculate": False},
    },
    missing_diagrams={"bar": True, "matrix": True, "heatmap": False},
    progress_bar=False,
)
report_path = REPORT_DIR / "profiling_raw.html"
# a barra de progresso por variável (tqdm, em stderr) ignora progress_bar=False
with open(os.devnull, "w") as devnull, contextlib.redirect_stderr(devnull):
    profile.to_file(report_path)
report_path.relative_to(ROOT)

WindowsPath('reports/profiling_raw.html')

## Alertas do relatório

Lista em texto dos alertas automáticos, para comparar com os pressupostos de `docs/decisions.md` sem abrir o HTML.

In [4]:
alerts = profile.get_description().alerts
pd.DataFrame(
    [(a.column_name, a.alert_type.name) for a in alerts],
    columns=["coluna", "alerta"],
).sort_values(["alerta", "coluna"]).reset_index(drop=True)

,coluna,alerta
0,citoglipton,CONSTANT
1,examide,CONSTANT
2,acetohexamide,HIGH_CORRELATION
3,change,HIGH_CORRELATION
4,diabetesMed,HIGH_CORRELATION
5,glimepiride-pioglitazone,HIGH_CORRELATION
6,glipizide-metformin,HIGH_CORRELATION
7,glyburide-metformin,HIGH_CORRELATION
8,insulin,HIGH_CORRELATION
9,max_glu_serum,HIGH_CORRELATION


## Leitura dos alertas

O relatório **confirma** o que já estava na EDA e em `docs/`; não traz nada que obrigue a rever uma decisão.

- **CONSTANT** - `examide`, `citoglipton`: 0 encontros com valor diferente de "No". Já registado em `docs/anexo_medicamentos.md` e `docs/dicionario_variaveis.md` (saem por serem constantes, T1.9).
- **HIGH_CORRELATION com V = 1,00** (`weight`, `payer_code`, `race` com `acetohexamide`, `troglitazone`, `glimepiride-pioglitazone`, `metformin-rosiglitazone`, `metformin-pioglitazone`, `glipizide-metformin`, `miglitol`, `max_glu_serum`): **artefacto**. Estes fármacos têm 1 a 38 encontros fora de "No" e o V de Cramér chega a 1 com células quase vazias. Não é associação real.
- **HIGH_CORRELATION real** - `change` × `insulin` (0,64), `diabetesMed` × `insulin` (0,59), `change` × `diabetesMed` (0,51): redundância esperada (as três descrevem a medicação); a ter em conta na seleção de features (T1.8).
- `readmitted` × `target`: trivial, o target é derivado de `readmitted`.
- **IMBALANCE** - a maioria dos medicamentos e os IDs de alta/origem: concentrados numa categoria; coerente com as prevalências do anexo de medicamentos e com P4.
- **MISSING** - `weight` (97%), `medical_specialty` (49%), `payer_code` (40%), `race`, `diag_3`: bate com P1 e com a §3 do `01_eda`. `A1Cresult` e `max_glu_serum` **não** aparecem como missing: "None" ficou como categoria (P2).
- **ZEROS / SKEWED** - `number_outpatient/emergency/inpatient`, `num_procedures`: maioria a 0, cauda longa; coerente com a §6 do `01_eda`.

Para o relatório final: a lista de alertas pode ir em anexo, com a nota sobre os falsos positivos do V de Cramér em categorias raras.